In [ ]:
# NOTEBOOK PARAMETERS
selected_model = 'neptune_1'
ensemble_size = 8
years = [2019,2020,2021]

In [ ]:
import matplotlib.pyplot as plt
from tqdm import tqdm
import pandas as pd
import xarray as xr
import numpy as np
import os
import sys

sys.path.append('library')
from atmocean.utils import remove_ocean_variables_suffix
from atmocean.visualize import (
    get_sst_bias_colormap,
    plot_dmi_timeseries,
    plot_map_on_grid,
    discrete_cmap,
    plot_rects,
)
from atmocean.enso import (
    monthly_resample_with_prediction_timedelta,
    monthly_groupby_with_prediction_timedelta,
    compute_indian_ocean_dipole_index,
    compute_iod_bias_sst,
    IOD_REGIONS,
    dmi,
)
from atmocean.utils import clean_forecasts
os.system('hostname')
%load_ext autoreload
%autoreload 2

In [ ]:
# get the forecasting horizon for this run
start_date = '01-01'
end_date = '12-31'
lead_times = [0, 10, 20, 30, 40, 50, 60]

main_dir = 'PATH-TO-MAIN-DIRECTORY'

data_dir = os.path.join(main_dir, 'data')
forecast_dir = os.path.join(data_dir, 'forecasts')

if 'highres' not in selected_model:
    climatology_src = os.path.join(forecast_dir, f'climatology-from_1993_to_2017.zarr')
else:
    climatology_src = os.path.join(forecast_dir, f'climatology_025-from_1993_to_2017.zarr')

if ensemble_size is None:
    forecasts_srcs = [
        os.path.join(forecast_dir, f'{selected_model}-from_{year}-{start_date}_to_{year}-{end_date}.zarr')
        for year in years
    ]
    out_img_dir = os.path.join(main_dir, f'validation/img/{selected_model}/s2s-{years[0]}-{years[-1]}/ocean_indices/iod')
else:
    forecasts_srcs = [
        os.path.join(forecast_dir, f'{selected_model}-from_{year}-{start_date}_to_{year}-{end_date}-ENS-{ensemble_size:02d}.zarr')
        for year in years
    ]
    out_img_dir = os.path.join(main_dir, f'validation/img/{selected_model}/s2s-{years[0]}-{years[-1]}-ENS-{ensemble_size:02d}/ocean_indices/iod')

os.makedirs(out_img_dir, exist_ok=True)
plot_kwargs = dict(dpi=100, bbox_inches='tight')
save_figures = True

In [ ]:
from dask.distributed import Client, LocalCluster, progress, wait
from tqdm import tqdm
import numpy as np
import dask
import ctypes

# spill to disk before hitting hard memory limits (avoids KilledWorker)
dask.config.set({
    'distributed.worker.memory.target': 0.6,
    'distributed.worker.memory.spill': 0.7,
    'distributed.worker.memory.pause': 0.8,
    'distributed.worker.memory.terminate': 0.95,
})

cpu_count = os.cpu_count() or 16
n_workers = min(16, cpu_count // 2)
threads_per_worker = 2
memory_limit = 948 // n_workers

print(f'memory_limit {memory_limit}GB')
print(f'n_workers {n_workers}')
print(f'threads_per_worker {threads_per_worker}')
cluster = LocalCluster(
    n_workers=n_workers, threads_per_worker=threads_per_worker,
    memory_limit=f'{memory_limit}GB',
    local_directory=os.path.join(main_dir, 'tmp_dask'),  # spill directory
)
client = Client(cluster)

def _trim_memory() -> int:
    libc = ctypes.CDLL("libc.so.6")
    return libc.malloc_trim(0)

client.run(_trim_memory)
client

In [ ]:
# open forecasts
forecasts = xr.open_mfdataset(
    forecasts_srcs, chunks={}, parallel=True, engine='zarr', preprocess=clean_forecasts,
    combine='nested', concat_dim='time', data_vars='minimal', coords='minimal', compat='override',
)
# manipulate the time dimension to align
if 'highres' not in selected_model:
    forecasts = forecasts.isel(prediction_timedelta=lead_times)
forecasts = forecasts.isel(depth=0, level=0, missing_dims='ignore')
# remove ocean variables suffix (oras, cglo, glor)
forecasts = remove_ocean_variables_suffix(forecasts)

In [ ]:
# open historical climatology
climatology = xr.open_zarr(climatology_src, chunks={})
if 'highres' not in selected_model:
    climatology = climatology.isel(prediction_timedelta=lead_times)
climatology = climatology.isel(depth=0, level=0, missing_dims='ignore')
# remove ocean variables suffix (oras, cglo, glor)
climatology = remove_ocean_variables_suffix(climatology)

In [ ]:
if 'highres' in selected_model:
    forecasts['longitude'] = climatology.longitude.values

# select only SST
forecasts = forecasts['thetao']
climatology = climatology['thetao']
# resample over time dimension
forecasts = monthly_resample_with_prediction_timedelta(forecasts) # = forecasts.resample({'time':'1M'}, skipna=True).mean(dim='time')
climatology = monthly_groupby_with_prediction_timedelta(climatology) # = climatology.groupby('time.month').mean(dim='time')

In [ ]:
forecasts, climatology = client.compute([forecasts, climatology])
progress([forecasts, climatology])
wait([forecasts, climatology])
# 13m 14s -> only amount of time needed for this notebook.

In [ ]:
forecasts = forecasts.result().load() if hasattr(forecasts, 'result') else forecasts.load()
climatology = climatology.result().load() if hasattr(climatology, 'result') else climatology.load()

In [ ]:
climatology['prediction_timedelta'] = pd.to_timedelta(climatology['prediction_timedelta'], unit='days')

In [ ]:
forecasts_dmi_east = dmi(ds=forecasts, clm=climatology, region=IOD_REGIONS['east'], spatial_avg=True)
forecasts_dmi_west = dmi(ds=forecasts, clm=climatology, region=IOD_REGIONS['west'], spatial_avg=True)

In [ ]:
if ensemble_size is None:
    forecasts_dmi_east = forecasts_dmi_east.load()
    forecasts_dmi_west = forecasts_dmi_west.load()
else:
    forecasts_dmi_east = forecasts_dmi_east.load().mean(dim='ensemble_member', skipna=True)
    forecasts_dmi_west = forecasts_dmi_west.load().mean(dim='ensemble_member', skipna=True)
forecasts_dmi = (forecasts_dmi_west - forecasts_dmi_east)

In [ ]:
colors = discrete_cmap(len(lead_times)+2, 'Blues_r')
colors = [colors(i) for i in range(len(lead_times))]

In [ ]:
seasonal_colors = {
    f'{lead_time}': colors[i]
    for i, lead_time in enumerate(lead_times)
}
seasonal_names = {
    f'{lead_time}': 'Reanalysis' if lead_time == 0 else f"Forecast t + {lead_time}"
    for lead_time in lead_times
}
models_data = {}
for i,lt in enumerate(lead_times):
    models_data[f'{lt}'] = forecasts_dmi.isel(prediction_timedelta=i)

In [ ]:
outfile = os.path.join(out_img_dir, f'iod_dmi_timeseries.png') if save_figures else None
plot_dmi_timeseries(
    data=models_data,
    colors=seasonal_colors,
    names=seasonal_names,
    dmi='IOD',
    outfile=outfile,
)

In [ ]:
(
    forecasts_anom, west_index, east_index, iod_dmi, iod_dmi_son, posy, negy
) = compute_indian_ocean_dipole_index(
    ds=forecasts,
    clm=climatology,
    regions=[IOD_REGIONS['west'], IOD_REGIONS['east']],
    is_lat_reversed=False,
    relax_constraints=True,
)

In [ ]:
print(f'Computing IOD SST Bias of atlas')
bias_sst, pos_sst, neg_sst = compute_iod_bias_sst(forecasts_anom, posy, negy, [9,10,11])
bias_sst, pos_sst, neg_sst = bias_sst.compute(), pos_sst.compute(), neg_sst.compute()

In [ ]:
from matplotlib.ticker import ScalarFormatter
from matplotlib.gridspec import GridSpec
import matplotlib.patches as mpatches
from matplotlib import pyplot as plt
import matplotlib.ticker as mticker
import cartopy.feature as cfeature
import cartopy.crs as ccrs

In [ ]:
grid_list = [
    'gs[0,0:2]',
    'gs[0,2:4]',
    'gs[0,4:6]',
    'gs[1,0:2]',
    'gs[1,2:4]',
    'gs[1,4:6]',
    'gs[2,2:4]',
]

In [ ]:
cmap, norm, levels = get_sst_bias_colormap(-3.0, 3.0)

# Box coordinates of IOD regions
west_box_coo = (50, -10, 20, 20) # lon_min, lat_min, width, height
east_box_coo = (90, -10, 20, 10) # lon_min, lat_min, width, height

central_longitude = 90
plot_extent = [30, 130, -25, 25] # centered on Indian Ocean region

pos_sst_list = []
for i,lt in enumerate(lead_times):
    if ensemble_size is None:
        pos_data = pos_sst.isel(prediction_timedelta=i).sel(latitude=slice(plot_extent[3], plot_extent[2]), longitude=slice(plot_extent[0], plot_extent[1]))
    else:
        pos_data = pos_sst.isel(prediction_timedelta=i).sel(latitude=slice(plot_extent[3], plot_extent[2]), longitude=slice(plot_extent[0], plot_extent[1])).mean(dim='ensemble_member', skipna=True)
    pos_sst_list.append(pos_data)

lats, lons = pos_sst_list[0].latitude.data, pos_sst_list[0].longitude.data

fig, axes = plt.subplots(1, 1, figsize=(15,5.5))
# period = season if season != 'all' else 'Annual'
fig.suptitle(f'Indian Ocean Dipole Index Spatial Composite SST', fontsize=16, y=1.05)

axes.axis('off')

# initialize grid spec
gs = GridSpec(3, 6, figure=fig)

axs = []

for i,lt in enumerate(lead_times):
    corr = xr.corr(pos_sst_list[0], pos_sst_list[i], dim=['latitude', 'longitude'])
    corr = np.around(float(corr.values),2)
    im, ax = plot_map_on_grid(
        bias=pos_sst_list[i], grid=eval(grid_list[i]), title='Reanalysis' if i == 0 else f'Forecast t+{lt}' + f' (r={corr})', 
        fig=fig, lons=lons, lats=lats, central_longitude=central_longitude, 
        cmap=cmap, norm=norm, levels=levels, 
    )
    ax.set_extent(plot_extent, crs=ccrs.PlateCarree())
    plot_rects(ax, [west_box_coo, east_box_coo], ['tab:blue', 'tab:red'])
    axs.append(ax)

# single colorbar
cax = fig.add_axes([0.25, -0.02, 0.5, 0.03]) # [left, bottom, width, height]
cbar = fig.colorbar(
    im, ax=axs, cax=cax, 
    orientation='horizontal', pad=0.05, 
    aspect=60, shrink=0.9, label='°C', 
    format=ScalarFormatter(), 
)
# adjust the plot
plt.subplots_adjust(
    left=0.05, right=0.95, bottom=0.1, top=0.9, wspace=-0.75, hspace=0.5
)

# save or plot figure
outfile = os.path.join(out_img_dir, f'iod_spatial_composite_pos.png')
plt.savefig(outfile, **plot_kwargs) if outfile else plt.show()

In [ ]:
cmap, norm, levels = get_sst_bias_colormap(-3.0, 3.0)

# Box coordinates of IOD regions
west_box_coo = (50, -10, 20, 20) # lon_min, lat_min, width, height
east_box_coo = (90, -10, 20, 10) # lon_min, lat_min, width, height

central_longitude = 90
plot_extent = [30, 130, -25, 25] # centered on Indian Ocean region

bias_sst_list = []
for i,lt in enumerate(lead_times):
    if ensemble_size is None:
        bias_data = bias_sst.isel(prediction_timedelta=i).sel(latitude=slice(plot_extent[3], plot_extent[2]), longitude=slice(plot_extent[0], plot_extent[1]))
    else:
        bias_data = bias_sst.isel(prediction_timedelta=i).sel(latitude=slice(plot_extent[3], plot_extent[2]), longitude=slice(plot_extent[0], plot_extent[1])).mean(dim='ensemble_member', skipna=True)
    bias_sst_list.append(bias_data)

lats, lons = bias_sst_list[0].latitude.data, bias_sst_list[0].longitude.data

fig, axes = plt.subplots(1, 1, figsize=(15,5.5))
# period = season if season != 'all' else 'Annual'
fig.suptitle(f'Indian Ocean Dipole Index Spatial Composite SST', fontsize=16, y=1.05)

axes.axis('off')

# initialize grid spec
gs = GridSpec(3, 6, figure=fig)

axs = []

for i,lt in enumerate(lead_times):
    corr = xr.corr(bias_sst_list[0], bias_sst_list[i], dim=['latitude', 'longitude'])
    corr = np.around(float(corr.values),2)
    im, ax = plot_map_on_grid(
        bias=bias_sst_list[i], grid=eval(grid_list[i]), title='Reanalysis' if i == 0 else f'Forecast t+{lt}' + f' (r={corr})', 
        fig=fig, lons=lons, lats=lats, central_longitude=central_longitude, 
        cmap=cmap, norm=norm, levels=levels, 
    )
    ax.set_extent(plot_extent, crs=ccrs.PlateCarree())
    plot_rects(ax, [west_box_coo, east_box_coo], ['tab:blue', 'tab:red'])
    axs.append(ax)

# single colorbar
cax = fig.add_axes([0.25, -0.02, 0.5, 0.03]) # [left, bottom, width, height]
cbar = fig.colorbar(
    im, ax=axs, cax=cax, 
    orientation='horizontal', pad=0.05, 
    aspect=60, shrink=0.9, label='°C', 
    format=ScalarFormatter(), 
)
# adjust the plot
plt.subplots_adjust(
    left=0.05, right=0.95, bottom=0.1, top=0.9, wspace=-0.75, hspace=0.5
)

# save or plot figure
outfile = os.path.join(out_img_dir, f'iod_spatial_composite_bias.png')
plt.savefig(outfile, **plot_kwargs) if outfile else plt.show()

In [ ]:
if '_025' in selected_model:
    sealand_mask = np.load(os.path.join(main_dir, 'data/masks/sealand_025_deg/sealand_mask_2d.npy')).astype(bool)
else:
    sealand_mask = np.load(os.path.join(main_dir, 'data/masks/sealand_1_deg/sealand_mask_2d.npy')).astype(bool)
sealand_mask = sealand_mask[np.newaxis]
sealand_mask = np.repeat(sealand_mask, repeats=len(lead_times), axis=0)
sealand_mask = np.repeat(sealand_mask, repeats=ensemble_size, axis=1)

grid_bias_list = ['gs[0,0:2]','gs[0,2:4]','gs[0,4:6]','gs[0,6:8]','gs[1,1:3]','gs[1,3:5]','gs[1,5:7]']
cmap, norm, levels = get_sst_bias_colormap(-3.0, 3.0)

central_longitude = 90
plot_extent = [30, 130, -25, 25] # centered on Indian Ocean region

bias_sst_list = []
bias_sst_masked = xr.where(~sealand_mask, np.nan, bias_sst)
for i,lt in enumerate(lead_times):
    if ensemble_size is None:
        bias_data = bias_sst_masked.isel(prediction_timedelta=i).sel(latitude=slice(plot_extent[3], plot_extent[2]), longitude=slice(plot_extent[0], plot_extent[1]))
    else:
        bias_data = bias_sst_masked.isel(prediction_timedelta=i).sel(latitude=slice(plot_extent[3], plot_extent[2]), longitude=slice(plot_extent[0], plot_extent[1])).mean(dim='ensemble_member', skipna=True)
    bias_sst_list.append(bias_data)

lats, lons = bias_sst_list[0].latitude.data, bias_sst_list[0].longitude.data

fig, axes = plt.subplots(1, 1, figsize=(13,4))
# period = season if season != 'all' else 'Annual'
# fig.suptitle(f'Indian Ocean Dipole Index Spatial Composite SST', fontsize=1, y=1.05)

axes.axis('off')

# initialize grid spec
gs = GridSpec(2, 8, figure=fig)

axs = []

for i,lt in enumerate(lead_times):
    corr = xr.corr(bias_sst_list[0], bias_sst_list[i], dim=['latitude', 'longitude'])
    corr = np.around(float(corr.values),2)
    im, ax = plot_map_on_grid(
        bias=bias_sst_list[i], grid=eval(grid_bias_list[i]), title='Ground Truth' if i == 0 else f't+{lt}' + f' (r={corr})', # title='Ground Truth' if i == 0 else f't+{i}Δ' + f' (r={corr})', 
        fig=fig, lons=lons, lats=lats, central_longitude=central_longitude, 
        cmap=cmap, norm=norm, levels=levels, draw_labels=dict(longitude=True, yloc=False) if i==0 else False,
    )
    ax.set_extent(plot_extent, crs=ccrs.PlateCarree())
    # plot_rects(ax, [west_box_coo, east_box_coo], ['tab:blue', 'tab:red'])
    axs.append(ax)

# single colorbar
cax = fig.add_axes([0.95, 0.05, 0.01, 0.9]) # [left, bottom, width, height]
cbar = fig.colorbar(
    im, ax=axs, cax=cax, orientation='vertical', label='°C',
)
# adjust the plot
plt.subplots_adjust(
    left=0.05, right=0.95, bottom=0.05, top=0.95, wspace=0.0, hspace=0.3
)

# save or plot figure
outfile = os.path.join(out_img_dir, f'iod_spatial_composite_bias.pdf')
plt.savefig(outfile, **plot_kwargs) if outfile else plt.show()